# analysis spaces, concretely
what varies when an interaction changes a query: its parameters, its structure, or what it's joined to. one sea-ice table, three instruments, then the same moves in other rooms.

data: nsidc sea ice index v4, regional monthly extent. `frac` = extent / that region's max, so 0.15 means "15% of the most ice this region ever had".

In [1]:
import numpy as np, pandas as pd, vibe_widget as vw
vw.config(model="google/gemini-3.8-flash")
si = pd.read_csv("data/sea_ice_regional_monthly.csv")
si.head()

,region,year,month,extent_km2,frac
0,Baffin,1978,11,745624.331,0.422
1,Baffin,1978,12,926473.071,0.525
2,Baffin,1979,1,1090625.815,0.618
3,Baffin,1979,2,1157634.866,0.656
4,Baffin,1979,3,1299583.477,0.736


## parameterization  ·  `WHERE region = :r AND month BETWEEN :m1 AND :m2 AND year BETWEEN :y1 AND :y2`
the brush is the where clause. it snaps to whole cells so the query it states is exact, and the answer rides on it.

In [2]:
w1 = vw.create(
    """heatmap of frac: years on x (1979..2026), months on y (1..12, January at top), one region at a time picked from a dropdown (start Barents). cells white at 0 to deep blue at 1, missing cells hatched.
    a window brush: drag on the heatmap to draw a rectangle that snaps to whole cells. drag its body to move it, drag an edge to resize, left/right arrow keys shift it one year, up/down one month.
    the mean frac of the cells inside rides on the brush's top edge as 'avg 0.42 · 12 cells'.
    under the chart, monospace, updating live: the equivalent SQL, e.g.
    SELECT AVG(frac) FROM ice WHERE region = 'Barents' AND month BETWEEN 6 AND 9 AND year BETWEEN 2005 AND 2015   -- 0.42""",
    si,
    theme="minimal",
    outputs=vw.outputs(where="dict region, month_lo, month_hi, year_lo, year_hi", avg_frac="mean frac inside the brush"),
)

## structuring  ·  `GROUP BY region, year HAVING AVG(frac) < 0.15`
the threshold is the having clause. one line shared across facets asks one question; a line per facet asks fourteen.

In [3]:
w2 = vw.create(
    """september only (a small month picker top-right to change). small multiples: one panel per region, 14 panels in rows of 5, each a bar chart of frac by year (1979..2026), thin bars.
    a horizontal threshold line shared across all panels, draggable (give it a wide invisible grab area, ~14px, so it is easy to catch), starts at 0.15; while dragging, read the pointer with d3.pointer(event, that panel's plot group), invert that panel's y scale and clamp to [0, 1]. bars under it turn red; each panel title ends with '· n of N years under'.
    a toggle 'shared / per panel': per panel gives every panel its own draggable line, each initialised to the shared line's current value.
    under the grid, monospace, live: SELECT region, year, AVG(frac) FROM ice WHERE month = 9 GROUP BY region, year HAVING AVG(frac) < 0.15   -- 61 rows""",
    si,
    theme="minimal",
    outputs=vw.outputs(under="dict region -> list of years under that panel's threshold", thresholds="dict region -> threshold value"),
)

## augmentation  ·  `JOIN co2 ON year`, and a fit computed outside the chart
the band is a filter on residuals. the model came from numpy, the co2 from mauna loa; the chart just lets you hold the band.

In [4]:
sept = si[si.month == 9].groupby(["region", "year"], as_index=False).frac.mean()
co2 = pd.read_csv("data/co2_mm_mlo.csv").groupby("year", as_index=False).average.mean().rename(columns={"average": "co2_ppm"})
joined = sept.merge(co2, on="year")
fits = {r: np.polyfit(g.co2_ppm, g.frac, 1) for r, g in joined.groupby("region")}     # co2 as the clock, not a causal claim
joined["pred"] = [np.polyval(fits[r], c) for r, c in zip(joined.region, joined.co2_ppm)]
joined["resid"] = (joined.frac - joined.pred).round(4)
joined.head()

,region,year,frac,co2_ppm,pred,resid
0,Baffin,1979,0.038,336.835000,0.040954,-0.0030
1,Baffin,1980,0.026,338.762500,0.040543,-0.0145
2,Baffin,1981,0.032,340.120000,0.040254,-0.0083
3,Baffin,1982,0.050,341.478333,0.039965,0.0100
4,Baffin,1983,0.085,343.152500,0.039608,0.0454


In [ ]:
w3 = vw.create(
    """scatter for one region (dropdown, start Barents): x co2_ppm, y frac (september) with the y axis fitted to the data of the shown region (not 0..1), one dot per year labelled with the year on hover, the fitted line drawn through the pred values.
    two draggable band edges parallel to the fitted line, starting at ±1 sd of resid. dots outside the band are filled dark orange, inside are light grey; their years are listed to the right sorted by |resid|, largest first.
    drag either edge: the edge follows the pointer, so dragging it away from the fitted line widens the band and toward the line narrows it; shift+drag moves both together. under the chart, with clear space below the x-axis label, monospace, live:
    SELECT s.year, s.frac, c.co2_ppm FROM sept s JOIN co2 c ON s.year = c.year WHERE region = 'Barents' AND ABS(frac - pred) > 0.061   -- 9 rows""",
    joined,
    theme="minimal",
    outputs=vw.outputs(outside="years outside the band for the shown region", band="dict lo, hi in resid units"),
)

## same moves, other rooms

### math · initial conditions are query points
a phase portrait is a field. a click is a query into it. keep the click and you can compare basins.

In [6]:
w4 = vw.create(
    """phase portrait of a damped pendulum, θ'' = -sin(θ) - c·θ'. x is θ in [-2π, 2π], y is ω in [-5, 5]. faint grey direction field arrows on a grid.
    click anywhere to drop a starting point: a persistent numbered dot; its trajectory (rk4, dt 0.02, 25 s) is drawn in that dot's colour. drag a dot and its trajectory re-integrates live. double-click a dot to remove it. up to 12 dots.
    a slider for damping c in [0, 1.5], start 0.25; all trajectories update as it moves.
    a small table right of the plot: dot number, θ0, ω0, which equilibrium it ends near (θ = 0, ±2π), and the time to get within 0.05 of it.""",
    theme="minimal",
    outputs=vw.outputs(points="list of {theta0, omega0}", damping="c"),
)

### the click, played back
the portrait keeps the initial conditions; a second widget re-integrates them and swings the bob. drop or drag a dot on the left, the replay restarts from it.

In [7]:
vw.config(model="anthropic/claude-opus-5")   # three.js + animation loop + input reactivity
w4b = vw.create(
    """3d damped pendulum replaying the trajectories of the phase portrait, three.js with OrbitControls. a fixed pivot, a thin rod, a bob; θ = 0 hangs straight down, θ grows counter-clockwise seen from the front. faint ground disk, camera a little above and in front.
    inputs: points is a list of {theta0, omega0} in radians and radians/s, one per numbered dot in the portrait, dot number = index + 1; damping is a number c. integrate θ'' = -sin(θ) - c·θ' yourself with rk4, dt 0.02, 25 s, exactly as the portrait does, so the swing matches the drawn curve. rod and bob take the portrait's colour for that dot (#d9534f, #2b7cb6, #2a9d8f, #e76f51, #8a508f, #f4a261, #457b9d, #b5179e, #3a0ca3, #38b000, #e63946, #4361ee by index).
    a row above the canvas: a segmented picker of dot numbers 1..n (n = points.length), play/pause, replay (back to t = 0), and 't = 3.4 s · θ 1.02 · ω -0.41' updating live. playback in real time, requestAnimationFrame, stops at 25 s.
    when points changes: find the index that was added or whose theta0/omega0 moved (compare old and new by index), select it, and play from t = 0. when damping changes: re-integrate the selected one and keep playing from the current t. if the selected index no longer exists, select the last dot. no points: bob at rest, picker and play disabled.""",
    theme="minimal",
    inputs=vw.inputs(points=w4.outputs.points, damping=w4.outputs.damping),
)
vw.config(model="google/gemini-3.8-flash")

Config(provider='openrouter', host='openrouter.ai', model='google/gemini-3.8-flash', key_source='OPENROUTER_API_KEY', environment='vscode-like', temperature=0.7, timeout=120.0, streaming=True, data_privacy='sample', sample_rows=3, mode='standard', theme=None, execution='auto', retry=2, agent_preset='project', agent_run=None, bypass_row_guard=False)

### medicine · a guideline cutoff you can argue with
synthetic cohort, shaped like ckd trajectories, no patients in it. clinicians disagree about cutoffs; dragging one shows how many people the argument is about.

In [8]:
rng = np.random.default_rng(7); n = 240
base, slope = rng.normal(78, 14, n), rng.normal(-2.2, 2.4, n)
pts = pd.DataFrame({"patient": np.repeat(np.arange(n), 8), "visit": np.tile(np.arange(8), n)})
pts["egfr"] = (base[pts.patient] + slope[pts.patient] * pts.visit + rng.normal(0, 3, len(pts))).round(1)
pts["age_group"] = np.where(rng.random(n) < 0.45, "65+", "under 65")[pts.patient]
pts.head()

,patient,visit,egfr,age_group
0,0,0,78.7,65+
1,0,1,74.6,65+
2,0,2,71.9,65+
3,0,3,68.3,65+
4,0,4,70.5,65+


In [9]:
w5 = vw.create(
    """egfr over visit, one thin grey line per patient, two panels side by side faceted by age_group (its two values are exactly 'under 65' and '65+'). y from 0 to 120.
    one horizontal guideline threshold shared by both panels (dragging it in either panel moves it in both), wide invisible grab area ~14px, starts at 60 (CKD stage 3 cutoff). text in the chart must not be selectable while dragging. while dragging, read the pointer y with d3.pointer(event, <the panel's plot group>) and invert the y scale, clamp to [0, 120]. a patient has 'crossed' when their egfr is below it at the last visit and was above it at the first; those lines turn orange.
    each panel header on one line with the title, e.g. 'under 65 · n crossed · median crossing visit k', small enough never to overlap. shift+click anywhere in a panel adds a second shared threshold at the clicked egfr value (at most two thresholds); headers then give counts per band. up/down arrow keys nudge the last-touched threshold by 1.""",
    pts,
    theme="minimal",
    outputs=vw.outputs(crossed="patient ids that crossed the first threshold", thresholds="list of threshold values"),
)

### earth science · a shape you draw feeds a model you own
see `terrain_erosion.ipynb`: paint a heightmap, the 3d view renders it, python erodes it, the view updates. the interaction is data entry for a shape you couldn't type.